# Poisoned Papers: run the screening-agent experiment

Runs the agent with two open models on Colab's free GPU (Runtime > Change runtime type > T4 GPU). No API key needed. About 30-40 minutes per model with the default sample.

In [ ]:
!git clone -q https://github.com/Zubair-DLCS/poisoned-papers.git
%cd poisoned-papers
!pip -q install -r requirements.txt

## 1. Start a local model server (Ollama)

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

In [ ]:
MODELS = ['qwen2.5:7b', 'llama3.1:8b']   # any chat models from ollama.com/library
for m in MODELS:
    !ollama pull {m}

## 2. Run the experiment for each model

Resumable: if Colab disconnects, run this cell again and it continues from the cache.

In [ ]:
for m in MODELS:
    !python src/run.py --provider ollama --model {m} --per-class 10

## 3. Results and figures

In [ ]:
!python src/guard.py
!python src/analyze.py
!python src/figures.py

In [ ]:
from IPython.display import Image, display
for f in ['fig_attack_success.png', 'fig_clean_cost.png', 'fig_guard.png']:
    display(Image(f'figures/{f}'))

## 4. Download the results

The zip contains the decisions, summary tables and figures.

In [ ]:
!zip -qr poisoned_papers_results.zip results figures -x 'results/cache_*'
from google.colab import files
files.download('poisoned_papers_results.zip')

## Optional: a hosted model instead of Ollama

Any OpenAI-compatible provider works (Groq, Gemini, OpenRouter, OpenAI). Free tiers have rate limits, so set `LLM_RPM` to the provider's requests-per-minute limit.

In [ ]:
import os
# os.environ['LLM_API_KEY'] = '...'          # paste your key here, never commit it
# os.environ['LLM_RPM'] = '25'
# !python src/run.py --provider groq --model <model name from the provider's list> --per-class 10